# Подготовка данных

## Считаем данные в исходном txt формате как csv

In [1]:
!pip install findspark
!pip install sh

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


Перезапустить kernel после установки если пакеты поставились заново!

In [2]:
import findspark
findspark.init()

### Список файлов для обработки

In [3]:
!hdfs dfs -ls

Found 2 items
drwxr-xr-x   - ubuntu hadoop          0 2025-02-14 09:58 .sparkStaging
drwxr-xr-x   - ubuntu hadoop          0 2025-02-14 07:55 data


In [5]:
!hdfs dfs -ls data

Found 25 items
-rw-r--r--   1 ubuntu hadoop 1879048192 2025-02-14 07:45 data/.distcp.tmp.attempt_local150647860_0001_m_000000_0.1739519107667
-rw-r--r--   1 ubuntu hadoop 2895460543 2025-02-14 07:44 data/2019-10-21.txt
-rw-r--r--   1 ubuntu hadoop 2994906767 2025-02-14 07:34 data/2020-01-19.txt
-rw-r--r--   1 ubuntu hadoop 2995176166 2025-02-14 07:43 data/2020-03-19.txt
-rw-r--r--   1 ubuntu hadoop 2996034632 2025-02-14 07:37 data/2020-04-18.txt
-rw-r--r--   1 ubuntu hadoop 2995666965 2025-02-14 07:40 data/2020-05-18.txt
-rw-r--r--   1 ubuntu hadoop 2995778382 2025-02-14 07:31 data/2020-09-15.txt
-rw-r--r--   1 ubuntu hadoop 2995868596 2025-02-14 07:37 data/2020-10-15.txt
-rw-r--r--   1 ubuntu hadoop 2995390576 2025-02-14 07:34 data/2021-01-13.txt
-rw-r--r--   1 ubuntu hadoop 2995780517 2025-02-14 07:39 data/2021-02-12.txt
-rw-r--r--   1 ubuntu hadoop 2995191659 2025-02-14 07:31 data/2021-03-14.txt
-rw-r--r--   1 ubuntu hadoop 3041980335 2025-02-14 07:40 data/2021-07-12.txt
-rw-r--r-- 

In [15]:
import re
import sh

def get_hdfs_files(directory:str, extension:str="txt") -> list:
    '''
    Params:
    directory: an HDFS directory e.g. /my/hdfs/location
    '''
    output = sh.hdfs('dfs','-ls',directory).split('\n')
    files = []
    for line in output:
        match = re.search(f'({re.escape(directory)}.*\.{extension}$)', line)
        if match:
            files.append(match.group(0))

    return files

files = get_hdfs_files(directory='data', extension='txt')
files


['data/2019-08-22.txt',
 'data/2019-09-21.txt',
 'data/2019-10-21.txt',
 'data/2019-11-20.txt',
 'data/2019-12-20.txt',
 'data/2020-01-19.txt',
 'data/2020-02-18.txt',
 'data/2020-03-19.txt',
 'data/2020-04-18.txt',
 'data/2020-05-18.txt',
 'data/2020-06-17.txt',
 'data/2020-07-17.txt',
 'data/2020-09-15.txt',
 'data/2020-10-15.txt',
 'data/2020-11-14.txt',
 'data/2020-12-14.txt',
 'data/2021-01-13.txt',
 'data/2021-02-12.txt',
 'data/2021-03-14.txt',
 'data/2021-04-13.txt',
 'data/2021-05-13.txt',
 'data/2021-06-12.txt',
 'data/2021-07-12.txt',
 'data/2021-08-11.txt',
 'data/2021-09-10.txt',
 'data/2021-10-10.txt',
 'data/2021-11-09.txt',
 'data/2021-12-09.txt',
 'data/2022-01-08.txt',
 'data/2022-02-07.txt',
 'data/2022-03-09.txt',
 'data/2022-04-08.txt',
 'data/2022-05-08.txt',
 'data/2022-06-07.txt',
 'data/2022-07-07.txt',
 'data/2022-08-06.txt',
 'data/2022-09-05.txt',
 'data/2022-10-05.txt',
 'data/2022-11-04.txt']

Создадим SparkSession

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("OTUS")
        .getOrCreate()
)

In [4]:
from pyspark.sql.types import StructType, StructField, LongType, TimestampType, IntegerType, DoubleType, BooleanType

schema = StructType([
    StructField("transaction_id", LongType(), True),
    StructField("tx_datetime", TimestampType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("terminal_id", IntegerType(), True),
    StructField("tx_amount", DoubleType(), True),
    StructField("tx_time_seconds", LongType(), True),
    StructField("tx_time_days", IntegerType(), True),
    StructField("tx_fraud", IntegerType(), True),
    StructField("tx_fraud_scenario", IntegerType(), True)
])

# Просмотр данных в файле
Загрузим данные из одного файла.

In [5]:
file_path = "data/2022-09-05.txt"
whole_data = "data/"

df = spark.read.csv(
    whole_data,
    header=False,  
    comment="#",  # comment character
    schema=schema, 
    sep=",",       # separator (comma in this case)
    mode="PERMISSIVE" # Handles lines with more or fewer columns.
)

In [6]:
df.show(5)     # Show the data

+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|        tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|             0|2019-08-22 06:51:03|          0|        711|    70.91|          24663|           0|       0|                0|
|             1|2019-08-22 05:10:37|          0|          0|    90.55|          18637|           0|       0|                0|
|             2|2019-08-22 19:05:33|          0|        753|    35.38|          68733|           0|       0|                0|
|             3|2019-08-22 07:21:33|          0|          0|    80.41|          26493|           0|       0|                0|
|             4|2019-08-22 09:06:17|          1|        981|   102.83|          32777|           0|       0|   

In [43]:
#df.describe().show()

KeyboardInterrupt: 

# Поиск незаполненных значений

In [44]:
df.columns

['transaction_id',
 'tx_datetime',
 'customer_id',
 'terminal_id',
 'tx_amount',
 'tx_time_seconds',
 'tx_time_days',
 'tx_fraud',
 'tx_fraud_scenario']

In [7]:
from pyspark.sql.functions import isnan, when, count, col, isnull

# columns = ['customer_id', 'terminal_id', 'tx_amount', 'tx_time_seconds', 'tx_time_days', 'tx_fraud', 'tx_fraud_scenario']
df.select([count(when(isnull(c), c)).alias(c) for c in df.columns]).show()

+--------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|             0|       3805|          0|      40312|        0|              0|           0|       0|                0|
+--------------+-----------+-----------+-----------+---------+---------------+------------+--------+-----------------+



Видим, что колонки tx_datetime, tx_amount содержат null значения.
Удалим такие данные - не будем обрабатывать их.

In [33]:
df_dropped = df.na.drop()

In [34]:
df_dropped.count()

46993395

In [40]:
(df.groupBy('tx_fraud_scenario')  # Group by 'customer_id'
   .count()                 # Count the number of rows per group
   .sort('count', ascending=False)  # Sort by the count in descending order
   .show())                 # Show the result

+-----------------+--------+
|tx_fraud_scenario|   count|
+-----------------+--------+
|                0|44449978|
|                2| 2453915|
|                3|   64226|
|                1|   25785|
+-----------------+--------+



Поделим данные на 10 разделов и сохраним в формате parquete

In [6]:
%%time
df.count()

CPU times: user 440 µs, sys: 3.42 ms, total: 3.87 ms
Wall time: 14.5 s


46987223

In [24]:
(
    df
        .repartition(10)
        .write
        .mode("overwrite")
        .parquet("parquet_data/data.parquet")
)

Проверим размер сохраненного файла

In [26]:
!hdfs dfs -ls -h parquet_data

Found 1 items
drwxr-xr-x   - ubuntu hadoop          0 2025-02-13 13:43 parquet_data/data.parquet


Сравним скорость посчета строк из parquet

In [27]:
df_from_parquet = spark.read.parquet("parquet_data/data.parquet")

In [28]:
%%time
df_from_parquet.count()

CPU times: user 2.38 ms, sys: 409 µs, total: 2.79 ms
Wall time: 8.94 s


1879794138

In [ ]:
df_from_parquet.select('customer_id').groupBy('customer_id').count().show()

TypeError: 'DataFrameNaFunctions' object is not callable

In [21]:
from pyspark.sql.functions import isnan, when, count, col

# df_from_parquet.select([count(when(isnan(c), c)).alias(c) for c in df.columns]).show()
df_from_parquet.select([count(when(isnan(c), c)).alias(c) for c in ['customer_id']]).show()

NameError: name 'df_from_parquet' is not defined